ипортируем библиотеки

In [1]:
import pandas as pd
import numpy as np
import requests

1. Читаем json

In [2]:
auto_df = pd.read_json("../data/auto.json")
pd.options.display.float_format = '{:.2f}'.format
print(auto_df)

        CarNumber  Refund    Fines    Make    Model
0    Y163O8161RUS       2  3200.00    Ford    Focus
1     E432XX77RUS       1  6500.00  Toyota    Camry
2     7184TT36RUS       1  2100.00    Ford    Focus
3    X582HE161RUS       2  2000.00    Ford    Focus
5    92918M178RUS       1  5700.00    Ford    Focus
..            ...     ...      ...     ...      ...
926  Y163O8161RUS       2  1600.00    Ford    Focus
927  M0309X197RUS       1 22300.00    Ford    Focus
928  O673E8197RUS       2   600.00    Ford    Focus
929  8610T8154RUS       1  2000.00    Ford    Focus
930  H419XE197RUS       2  8594.59  Toyota  Corolla

[725 rows x 5 columns]


In [3]:
missing_models = auto_df[auto_df["Model"].isna()]
print(missing_models)

        CarNumber  Refund    Fines        Make Model
39   M5039X197RUS       2  7400.00  Volkswagen   NaN
122  M589CH197RUS       2  7900.00  Volkswagen   NaN
195  H837YK197RUS       2  4200.00        Audi   NaN
236  E316EH197RUS       1  1300.00  Volkswagen   NaN
486  X023HY197RUS       2 10200.00       Volvo   NaN
554  X023HY197RUS       2  6800.00       Volvo   NaN
582  Y687HM197RUS       1  8594.59         BMW   NaN
724  Y693HM197RUS       2  6500.00         BMW   NaN
778  96907X197RUS       2  3000.00         BMW   NaN


2. обогащаем

In [4]:
sample_200 = auto_df.sample(
    n=200,
    replace=True,
    random_state=21,
)
rng = np.random.default_rng(21)
sample_200["Fines"] = rng.normal(
    loc=auto_df["Fines"].mean(),
    scale=auto_df["Fines"].std(),
    size=200
).clip(0)

sample_200["Refund"] = rng.integers(
    low=1,
    high=auto_df["Refund"].max() + 1,
    size=200
)
sample_200 = sample_200.reset_index(drop=True)
print(sample_200)

        CarNumber  Refund    Fines        Make   Model
0    Y351O8197RUS       2 14442.81        Ford   Focus
1     H917TC36RUS       1 33219.53        Ford   Focus
2    C589EY154RUS       1     0.00        Ford   Focus
3     K846YE77RUS       1 36087.40  Volkswagen  Passat
4    X4108H125RUS       2  7823.29        Ford   Focus
..            ...     ...      ...         ...     ...
195  M942OT152RUS       2 12897.29        Ford   Focus
196  Y187O8161RUS       2 18807.85        Ford   Focus
197  7064C8197RUS       2 14196.50  Volkswagen  Passat
198  8437XX154RUS       1     0.00        Ford   Focus
199   C410X938RUS       2 15308.58        Ford   Focus

[200 rows x 5 columns]


In [5]:
concat_rows = pd.concat([auto_df, sample_200], ignore_index=True)
print(concat_rows)

        CarNumber  Refund    Fines        Make   Model
0    Y163O8161RUS       2  3200.00        Ford   Focus
1     E432XX77RUS       1  6500.00      Toyota   Camry
2     7184TT36RUS       1  2100.00        Ford   Focus
3    X582HE161RUS       2  2000.00        Ford   Focus
4    92918M178RUS       1  5700.00        Ford   Focus
..            ...     ...      ...         ...     ...
920  M942OT152RUS       2 12897.29        Ford   Focus
921  Y187O8161RUS       2 18807.85        Ford   Focus
922  7064C8197RUS       2 14196.50  Volkswagen  Passat
923  8437XX154RUS       1     0.00        Ford   Focus
924   C410X938RUS       2 15308.58        Ford   Focus

[925 rows x 5 columns]


3. обогащение concat_rows

In [6]:
np.random.seed(21)
Year = pd.Series(np.random.randint(1980, 2020, concat_rows.shape[0]), name="Year")
fines = pd.concat([concat_rows, Year], axis=1)
print(fines)

        CarNumber  Refund    Fines        Make   Model  Year
0    Y163O8161RUS       2  3200.00        Ford   Focus  1989
1     E432XX77RUS       1  6500.00      Toyota   Camry  1995
2     7184TT36RUS       1  2100.00        Ford   Focus  1984
3    X582HE161RUS       2  2000.00        Ford   Focus  2015
4    92918M178RUS       1  5700.00        Ford   Focus  2014
..            ...     ...      ...         ...     ...   ...
920  M942OT152RUS       2 12897.29        Ford   Focus  1981
921  Y187O8161RUS       2 18807.85        Ford   Focus  1992
922  7064C8197RUS       2 14196.50  Volkswagen  Passat  2007
923  8437XX154RUS       1     0.00        Ford   Focus  2005
924   C410X938RUS       2 15308.58        Ford   Focus  1997

[925 rows x 6 columns]


4. обогащаем surname

In [7]:
surnames_df = pd.read_json("../../datasets/surname.json" )
surnames_df = surnames_df[1:].reset_index(drop=True)
surnames_series = surnames_df.iloc[:,0]
print(surnames_series)

0        ADAMS
1        ALLEN
2      ALVAREZ
3     ANDERSON
4       BAILEY
        ...   
95    WILLIAMS
96      WILSON
97        WOOD
98      WRIGHT
99       YOUNG
Name: 0, Length: 100, dtype: str


In [8]:
mask = surnames_series.str.contains(r'[^\w\s]', regex=True)
print(mask)

0     False
1     False
2     False
3     False
4     False
      ...  
95    False
96    False
97    False
98    False
99    False
Name: 0, Length: 100, dtype: bool


In [9]:
unique_car_nums = concat_rows["CarNumber"].unique()
    
selected_surnames = np.random.choice(surnames_series, size=unique_car_nums.shape[0], replace=True)
owners = pd.DataFrame({
    "CarNumber": unique_car_nums,
    "SURNAME": selected_surnames
})
print(owners)

        CarNumber SURNAME
0    Y163O8161RUS   BAKER
1     E432XX77RUS    CRUZ
2     7184TT36RUS  MARTIN
3    X582HE161RUS    REED
4    92918M178RUS  COOPER
..            ...     ...
526  O136HO197RUS  HOWARD
527  O22097197RUS   EVANS
528  M0309X197RUS  ROGERS
529  O673E8197RUS  WILSON
530  8610T8154RUS    DIAZ

[531 rows x 2 columns]


In [10]:
def gen_car_nums(n, seed):
    characters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    np.random.seed(seed)
    car_numbers = [ 
        ''.join(np.random.choice(list(characters), size=9)) + "RUS"
        for _ in range(n)
    ]
    return car_numbers

fines_add = pd.DataFrame({
    "CarNumber": pd.Series(gen_car_nums(5, 21)),
    "Refund":np.random.choice(fines["Refund"], size=5, replace=True),
    "Fines":np.random.choice(fines["Fines"], size=5, replace=True),        
    "Make":np.random.choice(fines["Make"], size=5, replace=True),
    "Model":np.random.choice(fines["Model"], size=5, replace=True),
    "Year":np.random.choice(fines["Year"], size=5, replace=True)
})

print(fines_add)
fines = pd.concat([fines, fines_add], ignore_index=True)
print(fines)

      CarNumber  Refund    Fines    Make  Model  Year
0  JPE98KIUURUS       1  8891.22    Ford  Focus  2001
1  MFSG8FQ91RUS       2 12700.00   Skoda  Focus  1994
2  7MSPGU02JRUS       1   500.00    Ford  Focus  2017
3  QAFF9EM44RUS       2  1000.00  Toyota  Focus  2013
4  MZ606ZPCHRUS       2   500.00    Ford  Focus  2008
        CarNumber  Refund    Fines    Make  Model  Year
0    Y163O8161RUS       2  3200.00    Ford  Focus  1989
1     E432XX77RUS       1  6500.00  Toyota  Camry  1995
2     7184TT36RUS       1  2100.00    Ford  Focus  1984
3    X582HE161RUS       2  2000.00    Ford  Focus  2015
4    92918M178RUS       1  5700.00    Ford  Focus  2014
..            ...     ...      ...     ...    ...   ...
925  JPE98KIUURUS       1  8891.22    Ford  Focus  2001
926  MFSG8FQ91RUS       2 12700.00   Skoda  Focus  1994
927  7MSPGU02JRUS       1   500.00    Ford  Focus  2017
928  QAFF9EM44RUS       2  1000.00  Toyota  Focus  2013
929  MZ606ZPCHRUS       2   500.00    Ford  Focus  2008

[93

In [11]:
print(len(owners))
owners = owners[:(len(owners)) - 20]

owners_add = pd.DataFrame({
    "CarNumber": pd.Series(gen_car_nums(3, 20)),
    "SURNAME": np.random.choice(owners["SURNAME"], size=3, replace=True)
})

print(owners_add)
owners = pd.concat([owners, owners_add], ignore_index=True)

print(owners)

531
      CarNumber SURNAME
0  90P520JULRUS   YOUNG
1  WH86V00TQRUS  PARKER
2  QH8G0NLZDRUS   BAKER
        CarNumber  SURNAME
0    Y163O8161RUS    BAKER
1     E432XX77RUS     CRUZ
2     7184TT36RUS   MARTIN
3    X582HE161RUS     REED
4    92918M178RUS   COOPER
..            ...      ...
509  O50197197RUS  JACKSON
510  7608EE777RUS   MILLER
511  90P520JULRUS    YOUNG
512  WH86V00TQRUS   PARKER
513  QH8G0NLZDRUS    BAKER

[514 rows x 2 columns]


In [12]:
merged_inner = pd.merge(fines, owners, on='CarNumber', how='inner')
print(merged_inner)

        CarNumber  Refund    Fines        Make   Model  Year SURNAME
0    Y163O8161RUS       2  3200.00        Ford   Focus  1989   BAKER
1     E432XX77RUS       1  6500.00      Toyota   Camry  1995    CRUZ
2     7184TT36RUS       1  2100.00        Ford   Focus  1984  MARTIN
3    X582HE161RUS       2  2000.00        Ford   Focus  2015    REED
4    92918M178RUS       1  5700.00        Ford   Focus  2014  COOPER
..            ...     ...      ...         ...     ...   ...     ...
898  M942OT152RUS       2 12897.29        Ford   Focus  1981    REED
899  Y187O8161RUS       2 18807.85        Ford   Focus  1992  ROGERS
900  7064C8197RUS       2 14196.50  Volkswagen  Passat  2007   PRICE
901  8437XX154RUS       1     0.00        Ford   Focus  2005  BAILEY
902   C410X938RUS       2 15308.58        Ford   Focus  1997  WALKER

[903 rows x 7 columns]


In [13]:
merged_outer = pd.merge(fines, owners, on='CarNumber', how='outer')
print(merged_outer)

        CarNumber  Refund    Fines  Make  Model    Year SURNAME
0    704687163RUS    2.00  1400.00  Ford  Focus 2004.00    WOOD
1    704787163RUS    2.00  2800.00  Ford  Focus 1992.00   PATEL
2    704987163RUS    2.00  8594.59  Ford  Focus 1985.00   LEWIS
3    705287163RUS    2.00  2000.00  Ford  Focus 1980.00   SMITH
4    705387163RUS    2.00   700.00  Ford  Focus 1987.00   MOORE
..            ...     ...      ...   ...    ...     ...     ...
928  Y969O8197RUS    2.00  7800.00  Ford  Focus 1992.00  MORGAN
929  Y973O8197RUS    2.00  8594.59  Ford  Focus 2005.00    KING
930  Y973O8197RUS    1.00 34800.00  Ford  Focus 2003.00    KING
931  Y973O8197RUS    1.00 69600.00  Ford  Focus 2017.00    KING
932  Y973O8197RUS    1.00  3058.41  Ford  Focus 2000.00    KING

[933 rows x 7 columns]


In [14]:
merged_fines = pd.merge(fines, owners, on='CarNumber', how='left')
print(merged_fines)

        CarNumber  Refund    Fines    Make  Model  Year SURNAME
0    Y163O8161RUS       2  3200.00    Ford  Focus  1989   BAKER
1     E432XX77RUS       1  6500.00  Toyota  Camry  1995    CRUZ
2     7184TT36RUS       1  2100.00    Ford  Focus  1984  MARTIN
3    X582HE161RUS       2  2000.00    Ford  Focus  2015    REED
4    92918M178RUS       1  5700.00    Ford  Focus  2014  COOPER
..            ...     ...      ...     ...    ...   ...     ...
925  JPE98KIUURUS       1  8891.22    Ford  Focus  2001     NaN
926  MFSG8FQ91RUS       2 12700.00   Skoda  Focus  1994     NaN
927  7MSPGU02JRUS       1   500.00    Ford  Focus  2017     NaN
928  QAFF9EM44RUS       2  1000.00  Toyota  Focus  2013     NaN
929  MZ606ZPCHRUS       2   500.00    Ford  Focus  2008     NaN

[930 rows x 7 columns]


In [15]:
merged_owners = pd.merge(fines, owners, on='CarNumber', how='right')
print(merged_owners)

        CarNumber  Refund    Fines   Make    Model    Year  SURNAME
0    Y163O8161RUS    2.00  3200.00   Ford    Focus 1989.00    BAKER
1    Y163O8161RUS    2.00  1600.00   Ford    Focus 1980.00    BAKER
2    Y163O8161RUS    1.00     0.00   Ford    Focus 2019.00    BAKER
3    Y163O8161RUS    1.00 18027.55   Ford    Focus 2017.00    BAKER
4    Y163O8161RUS    2.00     0.00   Ford    Focus 2017.00    BAKER
..            ...     ...      ...    ...      ...     ...      ...
901  O50197197RUS    2.00  7800.00   Ford    Focus 1992.00  JACKSON
902  7608EE777RUS    1.00  4000.00  Skoda  Octavia 2000.00   MILLER
903  90P520JULRUS     NaN      NaN    NaN      NaN     NaN    YOUNG
904  WH86V00TQRUS     NaN      NaN    NaN      NaN     NaN   PARKER
905  QH8G0NLZDRUS     NaN      NaN    NaN      NaN     NaN    BAKER

[906 rows x 7 columns]


5. Пиво на столе

In [16]:
pivot_table = fines.pivot_table(
    values='Fines',
    index=['Make', 'Model'],
    columns='Year',
    aggfunc='sum' 
)
print(pivot_table)

Year                   1980      1981      1982     1983     1984      1985  \
Make       Model                                                              
Ford       Focus   67569.34 464669.33 186784.03 88348.95 92113.44 145869.59   
           Mondeo       NaN       NaN       NaN      NaN      NaN       NaN   
Skoda      Focus        NaN       NaN       NaN      NaN      NaN       NaN   
           Octavia  8098.42       NaN   6900.00 11594.59 26600.88  10294.59   
Toyota     Camry   28743.35   8594.59       NaN  7200.00      NaN       NaN   
           Corolla      NaN       NaN   2000.00      NaN      NaN       NaN   
           Focus        NaN       NaN       NaN      NaN      NaN       NaN   
Volkswagen Golf    30900.00       NaN       NaN  8594.59   300.00  24000.00   
           Jetta        NaN       NaN       NaN      NaN      NaN       NaN   
           Passat       NaN  13696.36       NaN  3200.00 10000.00   5000.00   
           Touareg      NaN       NaN       NaN     

6. сохраняем

In [17]:
fines.to_csv("../data/fines.csv", index=False)
owners.to_csv("../data/owners.csv", index=False)

In [18]:
# concat_rows.count()

In [19]:
fines.count()

CarNumber    930
Refund       930
Fines        930
Make         930
Model        919
Year         930
dtype: int64